# BASELINE: Template IDs + LSTM (DeepLog-style)
# logs
#  template mining
#  event ids
#  sequences
#  LSTM
#  anomaly prediction

In [1]:
# Parsing logs

import csv
import json

dataset = []

with open("graylog_1.csv", newline="", encoding="utf-8") as f:

    reader = csv.DictReader(f)

    for row in reader:

        try:

            log_data = json.loads(row["log"])

            event = {

                "timestamp":
                    row["@timestamp"],

                "request_id":
                    log_data.get("request_id"),

                "service":
                    log_data.get("service_name"),

                "level":
                    log_data.get("level"),

                "message":
                    log_data.get("full_message")
            }

            dataset.append(event)

        except Exception:

            continue

print(len(dataset))

788092


In [2]:
# External logs added
import pandas as pd
import json

public_dataset = []

df = pd.read_csv("OpenStack_2k.log_structured.csv")

print(df.columns)
print(df.head())

Index(['LineId', 'Logrecord', 'Date', 'Time', 'Pid', 'Level', 'Component',
       'ADDR', 'Content', 'EventId', 'EventTemplate'],
      dtype='str')
   LineId                           Logrecord        Date          Time  \
0       1  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:00.008   
1       2  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:00.272   
2       3  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:01.551   
3       4  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:01.813   
4       5  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:03.091   

     Pid Level                       Component  \
0  25746  INFO  nova.osapi_compute.wsgi.server   
1  25746  INFO  nova.osapi_compute.wsgi.server   
2  25746  INFO  nova.osapi_compute.wsgi.server   
3  25746  INFO  nova.osapi_compute.wsgi.server   
4  25746  INFO  nova.osapi_compute.wsgi.server   

                                                ADDR  \
0  req-38101a0b-2096-447d-96ea-a692162415ae

In [3]:
for _, row in df.iterrows():

    event = {

        "timestamp":
            row.get("Time", ""),

        "request_id":
            str(row.get("Pid", "unknown")),

        "service":
            "openstack-service",

        "level":
            row.get("Level", "INFO"),

        "message":
            row.get("Content", "")
    }

    public_dataset.append(event)

print(len(public_dataset))
print(public_dataset[0])

2000
{'timestamp': '00:00:00.008', 'request_id': '25746', 'service': 'openstack-service', 'level': 'INFO', 'message': '10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2477829'}


In [4]:
dataset.extend(public_dataset)

In [5]:
# Build traces

from collections import defaultdict

traces = defaultdict(list)

for event in dataset:

    rid = event["request_id"]

    if rid:

        traces[rid].append(event)

for trace in traces.values():

    trace.sort(
        key=lambda x: x["timestamp"]
    )

print("TRACES:", len(traces))

TRACES: 100022


In [6]:
!pip install drain3


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
# Template mining

from drain3 import TemplateMiner

miner = TemplateMiner()

for event in dataset:

    log_text = (
        f"{event['service']} "
        f"{event['level']} "
        f"{event['message']}"
    )
    result = miner.add_log_message(log_text)

    event["event_id"] = result["cluster_id"]
    event["template"] = result["template_mined"]

config file not found: drain3.ini


In [8]:
# Build vocabulary

tokens = set()

for e in dataset:

    tokens.add(e["event_id"])

token2id = {

    t: i + 1
    for i, t in enumerate(tokens)
}

id2token = {

    i: t
    for t, i in token2id.items()
}

In [9]:
template_by_id = {}

for e in dataset:

    template_by_id[e["event_id"]] = e["template"]

In [10]:
# Build sequences

WINDOW_SIZE = 3

X = []
y = []

for trace in traces.values():

    seq = [

        token2id[e["event_id"]]

        for e in trace

        if "event_id" in e
    ]

    if len(seq) <= WINDOW_SIZE:
        continue

    for i in range(

        len(seq) - WINDOW_SIZE

    ):

        X.append(

            seq[i:i+WINDOW_SIZE]
        )

        y.append(

            seq[i+WINDOW_SIZE]
        )

print(len(X))

490028


In [11]:
# Train/test split

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
# Torch dataset

import torch
from torch.utils.data import Dataset

class LogDataset(Dataset):

    def __init__(

        self,
        X,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.long
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [13]:
# Data loaders

from torch.utils.data import DataLoader

train_loader = DataLoader(

    LogDataset(X_train, y_train),

    batch_size=32,

    shuffle=True
)

val_loader = DataLoader(

    LogDataset(X_val, y_val),

    batch_size=32
)

In [14]:
# LSTM model

import torch.nn as nn

class DeepLogLSTM(nn.Module):

    def __init__(

        self,
        vocab_size,
        embedding_dim=64,
        hidden_size=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(

            vocab_size + 1,
            embedding_dim
        )

        self.lstm = nn.LSTM(

            embedding_dim,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(

            hidden_size,
            vocab_size + 1
        )

    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [15]:
# Train

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

model = DeepLogLSTM(

    vocab_size=len(token2id)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(

    model.parameters(),
    lr=1e-3
)

In [16]:
EPOCH = 5

In [17]:
for epoch in range(EPOCH):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(

            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} "
        f"Loss={avg_loss:.4f}"
    )

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            pred = logits.argmax(dim=1)

            correct += (pred == y_batch).sum().item()
            total += y_batch.size(0)

    acc = correct / total

    print(
        f"Epoch {epoch+1} "
        f"Loss={avg_loss:.4f} "
        f"ValAcc={acc:.4f}"
    )

Epoch 1 Loss=0.0846
Epoch 1 Loss=0.0846 ValAcc=0.9781
Epoch 2 Loss=0.0751
Epoch 2 Loss=0.0751 ValAcc=0.9781
Epoch 3 Loss=0.0747
Epoch 3 Loss=0.0747 ValAcc=0.9779
Epoch 4 Loss=0.0743
Epoch 4 Loss=0.0743 ValAcc=0.9779
Epoch 5 Loss=0.0740
Epoch 5 Loss=0.0740 ValAcc=0.9780


In [18]:
# Predict next event

model.eval()

sample = torch.tensor(

    [X_val[0]],

    dtype=torch.long
).to(device)

with torch.no_grad():

    logits = model(sample)

pred = logits.argmax(dim=1).item()

print(
    "REAL TEMPLATE:",
    template_by_id[id2token[y_val[0]]]
)

print(
    "PREDICTED TEMPLATE:",
    template_by_id[id2token[pred]]
)

REAL TEMPLATE: billing-service INFO <*> <*>
PREDICTED TEMPLATE: billing-service INFO <*> <*>


In [20]:
torch.save(
    model.state_dict(),
    "aiops_baseline.pt"
)